# Clase 4 — JEV, el LLM para decisiones

> **¿Cómo pasamos de una instrucción a una decisión que otra aplicación pueda usar directamente?**

En las clases del Módulo 2 ya aprendiste a diseñar prompts: anatomía, few-shot, chain of thought y repositorios de prompts. Hoy damos un paso distinto. En lugar de pedirle a un LLM que **escriba** una respuesta y después parsearla, vamos a usar un modelo que **decide**: **Jev**, un modelo *System One* que devuelve decisiones tipadas con probabilidades.

Esta clase trabaja con OpenRouter. Primero recordamos brevemente el flujo *prompt-and-parse* y su costo. Después introducimos los modelos System One, exploramos la **Decisions API** de Jev y cerramos clasificando tickets de recepción con decisiones tipadas.

## Objetivos de aprendizaje

Al finalizar la clase deberías poder:

- Explicar qué es un modelo System One y en qué se diferencia de un LLM conversacional.
- Nombrar las tres primitivas de decisión (choice, noul, score) y qué devuelve cada una.
- Consultar a Jev por la Decisions API de OpenRouter con un contrato `state` + `questions`.
- Leer e interpretar `probabilities`, `confidence` y `noul`.
- Elegir umbrales para escalar a revisión humana según el costo de cada tipo de error.
- Comparar un LLM con system prompt frente a un modelo de decisión tipada (Jev).
- Identificar cuándo conviene un modelo System One, un LLM, o un híbrido de ambos.

## Recorrido de la clase

| Paso | Tema |
|---:|---|
| 1 | Preparar OpenRouter y probar una consulta |
| 2 | El problema: de texto libre a decisión accionable |
| 3 | Modelos System One: qué son y en qué se diferencian de un LLM |
| 4 | La Decisions API de Jev |
| 5 | Incertidumbre y umbrales |
| 6 | Laboratorio: Jev para clasificar tickets |
| 7 | Análisis y límites |
| 8 | Síntesis y puente con RAG |

## Cómo trabajar con este notebook

1. Ejecutá las celdas en orden.
2. Observá primero el resultado antes de cambiar las variables `TODO`.
3. Ingresá tu clave de OpenRouter en la celda indicada; se guarda en el entorno de esta sesión.
4. Cada consulta de este notebook usa OpenRouter y requiere conexión a Internet.
5. Usá solamente textos ficticios. No pegues historias clínicas ni datos personales reales.

## Glosario mínimo

| Término | Explicación breve |
|---|---|
| LLM | Modelo de lenguaje que genera texto libre |
| System One | Modelo que devuelve decisiones tipadas con probabilidades, sin texto libre |
| Choice | Pregunta que elige una opción de un conjunto definido |
| Noul | Pregunta de sí/no que devuelve la probabilidad de sí |
| Score | Pregunta que ubica la entrada en una escala ordenada |
| Confianza | Grado de concentración de la distribución de probabilidades |
| Revisión humana | Control de una persona antes de tomar una decisión |
| Prompt-and-parse | Flujo en el que un LLM genera texto y la aplicación lo parsea |

## 1. Antes de empezar: cuenta, clave y costos

OpenRouter funciona como una puerta de entrada a distintos modelos. Para esta clase necesitamos una cuenta y una clave de API. La clave se guarda como variable de entorno.

In [ ]:
import json
import os
import ssl
from urllib.request import Request, urlopen

import certifi

# La clave se lee del entorno, nunca se escribe en el código.
# Configurala con:  export OPENROUTER_API_KEY="tu-clave"
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
if not OPENROUTER_API_KEY:
    raise RuntimeError(
        "Falta la variable de entorno OPENROUTER_API_KEY. "
        "Configurala con: export OPENROUTER_API_KEY='tu-clave'"
    )
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL = "openai/gpt-6-luna"
SSL_CONTEXT = ssl.create_default_context(cafile=certifi.where())

# Configuración para Jev (modelo System One) a través de la Decisions API
OPENROUTER_DECISIONS_URL = "https://openrouter.ai/api/alpha/decisions"
MODEL_JEV = "typesafe/jev-1.13"

## 2. El problema: de texto libre a decisión accionable

En el Módulo 2 aprendiste a escribir prompts para que un LLM genere una respuesta. Cuando esa respuesta debe alimentar un sistema, el flujo habitual es *prompt-and-parse*:

```text
prompt → LLM escribe texto/JSON → la aplicación parsea y valida → acción
```

Funciona, pero tiene un costo: el modelo puede desviarse del formato, inventar campos o dar respuestas ambiguas. Cada desvío hay que detectarlo y manejarlo en el código.

Hoy exploramos la alternativa opuesta: en lugar de pedir que **escriba**, le pedimos a un modelo que **decida**. El resultado ya viene tipado y con probabilidades, listo para que tu código lo use.

### Una función simple para consultar el LLM (baseline)

Guardamos una función mínima para consultar un LLM conversacional. La usaremos solo como **baseline de contraste** en el laboratorio, no para re-enseñar prompting.

In [3]:
def consultar_modelo(messages, temperature=0.2, max_tokens=250, response_format=None):
    payload = {
        "model": MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "reasoning": {"effort": "none"},
    }
    if response_format is not None:
        payload["response_format"] = response_format

    solicitud = Request(
        OPENROUTER_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json",
        },
    )
    with urlopen(solicitud, context=SSL_CONTEXT, timeout=60) as respuesta:
        return json.load(respuesta)["choices"][0]["message"]["content"].strip()

## 3. Modelos System One: qué son y en qué se diferencian de un LLM

Un LLM conversacional **genera texto**: recibe un prompt y produce una secuencia de tokens. Es excelente para redactar, explicar y conversar, pero cuando necesitamos una decisión, el texto es solo un intermediario que después hay que interpretar.

Existe otra familia de modelos pensada para el caso opuesto: en lugar de generar texto, **deciden**. Se llaman **System One**, por la analogía con el "pensamiento rápido" de Kahneman. Un modelo System One recibe un estado y responde preguntas tipadas con probabilidades. No escribe prosa ni explica su razonamiento: devuelve una decisión que tu código puede usar directamente.

### Las tres posibilidades de decisión

| Primitiva | Pregunta que responde | Qué devuelve |
|---|---|---|
| **Choice** | ¿Cuál de estas opciones? | La opción elegida, una probabilidad por opción y una confianza |
| **Noul** | ¿Se cumple esta condición? | La probabilidad de que sea sí |
| **Score** | ¿Dónde cae en una escala ordenada? | Una posición ponderada, probabilidad por nivel y confianza |


### Un ejemplo por "primitives"

**`Choice` — ¿cuál de estas opciones?**

- **Pregunta:** ¿A qué categoría pertenece este ticket?
- **Opciones:** "Soporte técnico", "Ventas", "Facturación"
- **Probabilidades:** 0.2, 0.3, 0.5
- **Confianza:** 0.7

El modelo reparte la probabilidad entre las opciones. La más probable es **"Facturación"** (50%), seguida de "Ventas" (30%) y "Soporte técnico" (20%). La **confianza** resume qué tan concentrada está esa distribución: un valor de `0.7` indica que la decisión es razonablemente firme, aunque no perfecta.

**`Noul` — ¿se cumple esta condición?**

- **Pregunta:** ¿Este ticket requiere revisión humana?
- **Respuesta:** noul = 0.83

*noul* es la **probabilidad de que la respuesta sea sí**. Un valor de `0.83` indica que el modelo cree con un 83% de probabilidad que el ticket **sí** requiere revisión humana. Es una pregunta de sí/no: cuanto más cerca de 1, más seguro está de que la condición se cumple.

**`Score` — ¿dónde cae en una escala ordenada?**

- **Pregunta:** ¿Qué tan urgente es este ticket?
- **Escala:** 1 (baja) a 5 (crítica)
- **Resultado:** posición ponderada ≈ 4.2
- **Confianza:** 0.9

El modelo ubica la entrada en una escala ordenada y devuelve una posición ponderada, la probabilidad por nivel y una confianza. Un valor cercano a 5 sugiere urgencia alta, útil para priorizar colas de trabajo. La confianza alta (`0.9`) indica que la posición es estable y se puede usar con seguridad para ordenar la cola.

### El estado: la información que recibe el modelo

Un modelo System One no recibe un prompt de texto libre: recibe un **estado** (`state`), es decir, la información que describe la situación que queremos evaluar. El estado puede ser tan simple como un mensaje de texto o tan complejo como un conjunto de campos estructurados.

```t
state = {
  "ticket": "Necesito un turno con dermatología el martes por la tarde.",
  "cliente": "María López",
  "canal": "chat"
}
```

El modelo **procesa lenguaje natural**: entiende el texto del estado y lo interpreta. Pero a diferencia de un LLM conversacional, no responde con más texto: entrega **datos estructurados** (una decisión tipada con probabilidades) que tu código puede usar directamente.

### Enmarcar el problema con preguntas tipadas

Para que el modelo decida bien, hay que **enmarcar el problema**: definir con precisión qué preguntas queremos responder y cuáles son las respuestas posibles. Cada pregunta tipada es una forma de decirle al modelo *qué querés saber* y *entre qué opciones*.

- Si querés elegir entre categorías → usás una **choice** y definís las opciones.
- Si querés saber si se cumple una condición → usás un **noul**.
- Si querés ubicar algo en una escala → usás un **score**.

La calidad de la decisión depende tanto del modelo como de cómo enmarcamos el problema. Una pregunta ambigua o con opciones mal definidas produce decisiones poco confiables, igual que un prompt confuso produce respuestas confusas.

## 4. La Decisions API de Jev

**Jev** (`typesafe/jev-1.13`) es el primer modelo System One de TypeSafe y está disponible por OpenRouter. No se llama por el endpoint de chat completions, sino por la **Decisions API**, con un contrato distinto:

```json
{
  "model": "typesafe/jev-1.13",
  "state": { ... },        # el contenido a evaluar
  "questions": { ... }     # preguntas tipadas (choice, noul, score)
}
```

La respuesta trae una entrada por pregunta, con probabilidades y confianza, más el costo de la llamada:

```json
{
  "answers": {
    "intencion": { "type": "choice", "choice": "turno", "probabilities": {...}, "confidence": 1 },
    "requiere_revision": { "type": "noul", "noul": 0.21 }
  },
  "usage": { "input_tokens": 457, "output_tokens": 79, "cost": 0.000019194 }
}
```

In [4]:
def consultar_jev(state, questions, timeout=60):
    """Consulta a Jev (System One) por la Decisions API de OpenRouter.

    Devuelve el diccionario completo de la respuesta (answers + usage).
    Lanza RuntimeError si la petición falla o la respuesta no trae 'answers'.
    """
    payload = {
        "model": MODEL_JEV,
        "state": state,
        "questions": questions,
    }
    solicitud = Request(
        OPENROUTER_DECISIONS_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json",
        },
    )
    with urlopen(solicitud, context=SSL_CONTEXT, timeout=timeout) as respuesta:
        datos = json.load(respuesta)
    if "answers" not in datos:
        raise RuntimeError(f"Respuesta inesperada de Jev: {datos}")
    return datos

### Definimos las preguntas tipadas

Cada pregunta describe qué evaluar y qué criterios usar. `Choice` elige la intención; `Noul` decide si hace falta revisión humana.

In [5]:
PREGUNTAS_JEV = {
    "intencion": {
        "type": "choice",
        "instructions": "¿Cuál es la intención de esta solicitud de recepción?",
        "criteria": {
            "turno": "Solicitud de turno o cambio de turno.",
            "horarios": "Pregunta por horarios de atención.",
            "receta": "Solicitud de receta.",
            "consulta_medica": "Consulta clínica o urgencia.",
            "otro": "Cualquier otra cosa.",
        },
    },
    "requiere_revision": {
        "type": "noul",
        "instructions": "¿Esta solicitud requiere revisión humana?",
        "criteria": {
            "true": "Es una consulta clínica, urgencia o caso ambiguo.",
            "false": "Es una solicitud administrativa simple.",
        },
    },
}

ticket_ejemplo = "Buen dia, le escribe Maricarmen, queria saber si el Dr ya volvio porque me habia quedado pendiente saber si voy el martes por la tarde, quiero confirmar. Gracias."

respuesta_je = consultar_jev({"ticket": ticket_ejemplo}, PREGUNTAS_JEV)
print(json.dumps(respuesta_je, ensure_ascii=False, indent=2))

{
  "model": "typesafe/jev-1.13-20260917",
  "answers": {
    "intencion": {
      "type": "choice",
      "choice": "turno",
      "probabilities": {
        "horarios": 0.22,
        "consulta_medica": 0.04,
        "receta": 0,
        "turno": 0.55,
        "otro": 0.19
      },
      "confidence": 0.43
    },
    "requiere_revision": {
      "type": "noul",
      "noul": 0.21
    }
  },
  "usage": {
    "input_tokens": 480,
    "output_tokens": 79,
    "cost": 2.016e-05
  },
  "id": "gen-dec-1790867439-AeB4B5hC8w1RMPgpURH5",
  "provider": "TypeSafe"
}


### Para interpretar

- `intencion.choice` es la opción elegida; `intencion.probabilities` reparte la probabilidad entre las alternativas y `intencion.confidence` resume qué tan concentrada está esa distribución.
- `requiere_revision.noul` es la **probabilidad de que la respuesta sea sí**. Un valor bajo (p. ej. `0.21`) indica que Jev cree que no hace falta revisión humana.
- ¿Qué diferencia ves con la salida JSON del LLM? ¿Cuál te da más información para decidir si escalar a una persona?

## 5. Incertidumbre y umbrales

La incertidumbre es la señal más valiosa de un modelo System One. Un LLM te da una etiqueta sin decirte qué tan seguro está. Jev te da una **probabilidad**: si `noul` vale `0.96`, está casi seguro de que sí; si vale `0.51`, es un tiro al aire.

Esa señal es oro para decidir cuándo **escalar a una persona** en vez de actuar automáticamente. La confianza describe la distribución de las alternativas, no si el flujo es seguro: los umbrales se eligen según el costo de cada tipo de error.

### Regla práctica

```text
si noul >= umbral_alto  →  escalar a revisión humana
si confidence < umbral  →  escalar a revisión humana
si no                   →  actuar automáticamente
```

El umbral no es un número mágico: depende del dominio. Si el costo de un error es alto (salud, dinero, seguridad), el umbral baja. Si el costo es bajo (una etiqueta de categoría), el umbral puede subir.

In [6]:
# Umbrales de ejemplo. Ajustalos según el costo de cada tipo de error.
UMBRAL_REVISION = 0.5   # si noul >= este valor, escalamos
UMBRAL_CONFIANZA = 0.7  # si confidence < este valor, escalamos

def decidir_accion(respuesta_je):
    answers = respuesta_je["answers"]
    noul = answers["requiere_revision"]["noul"]
    confianza = answers["intencion"].get("confidence", 1.0)
    if noul >= UMBRAL_REVISION or confianza < UMBRAL_CONFIANZA:
        return "se aconseja revisión humana"
    return "no se requiere revisión"

print("Decisión para el ticket de ejemplo:", decidir_accion(respuesta_je))

Decisión para el ticket de ejemplo: se aconseja revisión humana


> **Nota:** Jev no reemplaza a un LLM. Si necesitás prosa, explicaciones o redacción, usá un modelo conversacional. Si necesitás una decisión tipada y predecible, Jev es una alternativa directa al paso "prompt + parse".

## 6. Laboratorio: enmarcar un caso de atención al cliente

Ahora te toca a vos. Vamos a trabajar con un caso ficticio de atención al cliente en un banco. Tu tarea es **enmarcar el problema**: definir las preguntas tipadas que Jev debería responder para decidir cómo atender el caso.

### El caso

> Laura escribe al banco: *"Hola, soy Laura. Quiero saber si me pueden aumentar el límite de mi tarjeta de crédito porque viajo al exterior la semana que viene. Además me llegó un cargo de $45.000 que no reconozco."*

### Tu tarea

Escribí **3 preguntas, una por primitiva**, que ayuden a decidir cómo atender este caso:

1. **Choice** — ¿qué opción elegir? Definí las opciones posibles.
2. **Noul** — ¿se cumple una condición? Definí la condición.
3. **Score** — ¿dónde cae en una escala? Definí la escala.

Pensá en qué necesita saber el sistema para decidir si atiende automáticamente, escala a una persona o prioriza el caso. Después compará tu enmarcado con el ejemplo de la celda siguiente.

In [7]:
# Ejemplo de consulta para el caso de un Banco

CASO_BANCO = {
    "mensaje": "Hola, soy Laura. Quiero saber si me pueden aumentar el límite de mi tarjeta de crédito porque viajo al exterior la semana que viene. Además me llegó un cargo de $45.000 que no reconozco.",
    "cliente": "Laura",
    "canal": "chat",
    "suscripcion": "premium",
}

# Reemplazá los "####" con tus propias instrucciones y criterios según tu criterio de negocio.

PREGUNTAS_BANCO = {
    "motivo": {
        "type": "choice",
        "instructions": "",
        "criteria": {
            "aumento_limite": " #### ",
            "reclamo_cargo": " #### ",
            "consulta_general": "Consulta general sobre productos o servicios.",
            "otro": "Cualquier otra cosa.",
        },
    },
    "requiere_revision": {
        "type": "noul",
        "instructions": " #### ",
        "criteria": {
            "true": " #### ",
            "false": " #### ",
        },
    },
    "urgencia": {
        "type": "score",
        "instructions": " #### ",
        "criteria": {
            "1": "Puede esperar.",
            "5": "Requiere atención inmediata.",
        },
    },
}

In [ ]:
# Consultamos a Jev con el enmarcado definido.
respuesta_banco = consultar_jev(CASO_BANCO, PREGUNTAS_BANCO)
print(json.dumps(respuesta_banco, ensure_ascii=False, indent=2))

In [ ]:
# Interpretamos la decisión de Jev.
answers = respuesta_banco["answers"]

print("Motivo:", answers["motivo"]["choice"])
print("Probabilidades:", answers["motivo"].get("probabilities"))
print("Confianza:", answers["motivo"].get("confidence"))
print("Probabilidad de revisión (noul):", answers["requiere_revision"]["noul"])
print("Urgencia:", answers["urgencia"])
print("Costo (USD):", respuesta_banco.get("usage", {}).get("cost"))

## 7. Análisis y límites

Los dos sistemas resuelven la misma tarea con contratos muy distintos. Antes de elegir uno, conviene pensar en qué se gana y qué se pierde con cada enfoque.

### Preguntas de análisis

- ¿En qué casos coincidieron LLM y Jev? ¿En cuáles se desacordaron?
- ¿Qué clase fue más difícil para cada modelo?
- ¿Qué información extra da Jev que el LLM no da? ¿Cómo usarías la confianza o el `noul` para decidir cuándo escalar a una persona?
- ¿Qué pasa si el LLM devuelve JSON inválido? ¿Y si Jev devuelve una categoría fuera del conjunto? ¿Cuál es más fácil de validar?
- ¿Cuándo preferirías un LLM con system prompt y cuándo un modelo System One?

### El patrón híbrido

Jev no explica su razonamiento. Si necesitás una justificación escrita, podés combinar ambos: **Jev decide** (con probabilidades) y un **LLM redacta** el motivo. O escalar los casos de baja confianza a una persona.

### Límites de esta comparación

- La muestra es didáctica y chica: no alcanza para declarar un ganador estadístico.
- Los contratos son distintos: el LLM genera texto que hay que parsear; Jev devuelve una decisión tipada. Comparar solo la etiqueta elegida deja afuera esa diferencia.

In [ ]:
# Caso clínico: Jev debería marcarlo como consulta_medica y con noul alto.
mensaje_salud = "Tengo fiebre y dolor fuerte desde anoche, ¿qué medicamento debería tomar?"
respuesta_salud = consultar_jev({"ticket": mensaje_salud}, PREGUNTAS_JEV)
answers = respuesta_salud["answers"]

print("Intención:", answers["intencion"]["choice"])
print("Confianza:", answers["intencion"].get("confidence"))
print("Probabilidades:", answers["intencion"].get("probabilities"))
print("Probabilidad de revisión humana (noul):", answers["requiere_revision"]["noul"])
print("Decisión:", decidir_accion(respuesta_salud))
print("Costo (USD):", respuesta_salud.get("usage", {}).get("cost"))

Intención: consulta_medica
Confianza: 0.99
Probabilidades: {'receta': 0.01, 'otro': 0, 'consulta_medica': 0.99, 'horarios': 0, 'turno': 0}
Probabilidad de revisión humana (noul): 0.96
Decisión: se aconseja revisión humana
Costo (USD): 1.9236e-05


## 8. Síntesis de la clase

### Hoy aprendimos que

- Un LLM **escribe**; un modelo System One **decide**.
- Las tres primitivas de decisión son choice, noul y score.
- Jev se consulta por la Decisions API con un contrato `state` + `questions`.
- La incertidumbre (probabilidades y confianza) ayuda a decidir cuándo escalar a una persona.
- Los umbrales se eligen según el costo de cada tipo de error, no son números mágicos.
- La validación local y la revisión humana siguen siendo parte del sistema, con cualquier modelo.
- Jev y un LLM pueden combinarse: Jev decide, el LLM redacta.

### Comprobación conceptual

1. ¿Qué diferencia hay entre un LLM con system prompt y un modelo System One como Jev?
2. ¿Qué devuelve cada primitiva (choice, noul, score)?
3. ¿Qué información extra da Jev (probabilidades/confianza) y cómo la usarías?
4. ¿Cómo elegís el umbral para escalar a revisión humana?
5. ¿Cuándo preferirías un LLM, cuándo Jev y cuándo un híbrido?
